# Location Notes

The original notes and sketches lacked detailed rulers to judge positions with much accuracy.
Those notes relied on building outlines copied and pasted into different level maps.


Using multiple Worldographer maps requires some care to assure staircases and chimneys align between levels.

This notebook has a class to help to show how the features in distinct levels align.
Usually these are stairwells and chimneys, but they can also be multi-story atria.
The idea is to define the position of a feature that spans levels in local coordinates for each of the levels it spans.
The difference between the local coordinates provides the offsets required be sure the features align.

Battlemap Scale: Square = 2m (6').

In [1]:
from dataclasses import dataclass
from typing import Any

@dataclass
class Point:
    x: int
    y: int

    def __sub__(self, other: Any) -> "Point":
        match other:
            case Point():
                return Point(self.x - other.x, self.y - other.y)
            case _:
                return NotImplemented
    def __rsub__(self, other: Any) -> "Point":
        match other:
            case Point():
                return Point(other.x - self.x, other.y - self.y)
            case _:
                return NotImplemented
    def __add__(self, other: Any) -> "Point":
        match other:
            case Point():
                return Point(self.x + other.x, self.y + other.y)
            case _:
                return NotImplemented
    __radd__ = __add__

@dataclass
class Rect:
    top_left: Point
    bottom_right: Point

    def __sub__(self, other: Any) -> "Rect":
        match other:
            case Rect():
                return Rect(self.top_left - other.top_left, self.bottom_right - other.bottom_right)
            case Point() as pt:
                return Rect(Point(self.top_left.x - pt.x, self.top_left.y - pt.y), Point(self.bottom_right.x - pt.x, self.bottom_right.y - pt.y))
            case _:
                return NotImplemented
    def __rsub__(self, other: Any) -> "Rect":
        match other:
            case Rect():
                return Rect(other.top_left - self.top_left, other.bottom_right - self.bottom_right)
            case Point() as pt:
                return Rect(Point(pt.x - self.top_left.x, pt.y - self.top_left.y), Point(pt.x - self.bottom_right.x, pt.y - self.bottom_right.y))
            case _:
                return NotImplemented
    def __add__(self, other: Any) -> "Rect":
        match other:
            case Rect():
                return Rect(self.top_left + other.top_left, self.bottom_right + other.bottom_right)
            case Point() as pt:
                return Rect(Point(self.top_left.x + pt.x, self.top_left.y + pt.y), Point(self.bottom_right.x + pt.x, self.bottom_right.y + pt.y))
            case _:
                return NotImplemented
    __radd__ = __add__

## East and South Caves

The E cave has an external entrance and a chimney that are widely separated.

The S cave is similar, but there was less ambiguity in the original sketches.

In [2]:
stairs_3_l3 = Rect(Point(10, 7), Point(10, 10))
chimney_l3 = Rect(Point(18, 12), Point(18, 12))

stairs_3_l2 = Rect(Point(12, 12), Point(12, 15))
l3_to_l2 = stairs_3_l3 - stairs_3_l2
l3_to_l2

Rect(top_left=Point(x=-2, y=-5), bottom_right=Point(x=-2, y=-5))

In [3]:
stairs_2_l2 = Rect(Point(3, 3), Point(3, 6))
stairs_1_l2 = Rect(Point(9, 3), Point(9, 6))

stairs_2_l1 = Rect(Point(3, 10), Point(3, 13))
stairs_1_l1 = Rect(Point(9, 10), Point(9, 13))
l2_to_l1 = stairs_2_l2 - stairs_2_l1
assert l2_to_l1 == stairs_1_l2 - stairs_1_l1
l2_to_l1

Rect(top_left=Point(x=0, y=-7), bottom_right=Point(x=0, y=-7))

In [4]:
cave_l1 = Rect(Point(16, 6), Point(19, 6))

In [5]:
chimney_l2 = chimney_l3 - l3_to_l2
chimney_l2

Rect(top_left=Point(x=20, y=17), bottom_right=Point(x=20, y=17))

In [6]:
chimney_l1 = chimney_l2 - l2_to_l1
chimney_l1

Rect(top_left=Point(x=20, y=24), bottom_right=Point(x=20, y=24))

## The Underkeep

There are five buildings with access to the underkeep.

-  New Keep
-  Central Tower
-  Old Keep
-  Gatehouse

Rooms are associated with one of two buildings: Central Tower rooms 1-13, and Old Keep rooms 1-5.

Additionally, the original design showed the Underkeep under the West Lodge, with no connection. This may be changed to add a trap that opens in the "Central Tower Level -2 Room 2."

In [7]:
new_keep_stairs = Rect(Point(10, 18), Point(12, 21))
new_keep = Rect(Point(1,2), Point(18, 33))

New Keep's Eastern wall (at x=18 locally) is adjacent to Central Tower's eastern wall (at x=2 locally).

New Keep's west wing north wall (at y=22 locally) is adjacent to Central Tower's southern wall (at y=17 locally).

New Keep x=19 == Central Tower x=1.  Add 18 to convert CT x into NK x

New Keep y=22 == Central Tower y=18.  Add 4 to convert CT y into NK y.

In [8]:
central_tower = Rect(Point(2, 2), Point(18, 17))
central_tower_stairs_1 = Rect(Point(6, 11), Point(7, 11))
central_tower_stairs_2 = Rect(Point(9, 16), Point(9, 17))
CT_offset = Point(18, 4)

In [9]:
central_tower_stairs_1 + CT_offset

Rect(top_left=Point(x=24, y=15), bottom_right=Point(x=25, y=15))

In [10]:
central_tower_stairs_2 + CT_offset

Rect(top_left=Point(x=27, y=20), bottom_right=Point(x=27, y=21))

Central Tower's north wall (at y=2 locally) is Old Keep's north wall (at y=2 locally).

Central Tower's east wall (at x=18 locally) aligns with East Lodge's west wall (at x=1 locally).
East Lodge's east wall is (at x=7 locally) is 12' (4m, 2 squares) from Old Keep's west wall (at x=2 locally).
Width of the East Lodge is 7.

Old Keep x=2 == Central Tower x=18 + 8 - 7. Add 25 to convert OK x-coordinates into CT x. Add 51 to convert to NK x.

Old Keep y=2 == Central Tower y=2.  Add 0 to convert to OK y-coordinates to CT y. Add 4 to converto to NK y.

In [11]:
old_keep = Rect(Point(2, 2), Point(14, 14))
old_keep_stairs = Rect(Point(5, 19), Point(7, 20))
OK_offset = Point(18+8+25-7, 4)
OK_offset

Point(x=44, y=4)

In [12]:
old_keep_stairs + OK_offset

Rect(top_left=Point(x=49, y=23), bottom_right=Point(x=51, y=24))

Gate House north wall (at y=4 locally) is 48' (16m, 8 squares) from Old Keep south wall (at y=14).

Gate House west wall (at x=2 locally) as -24' (8m, 4 squares) from Old Keep east wall (at x=14).
West wall is also at +30' (10m, 5 squares) from the East Lodge east wall.

This is tricky because the Gate House doesn't have a neat alignment with any other structure.

Gate House x=2 == Old Keep x=?

Gate House y=4 == Old Keep y=?

In [15]:
gate_house = Rect(Point(2, 4), Point(13, 13))
gate_house_stairs = Rect(Point(9, 2), Point(10, 2))
GH_offset = Point(18+8+25-7+4, 4+8+12)
GH_offset

Point(x=48, y=24)

In [16]:
gate_house_stairs + GH_offset

Rect(top_left=Point(x=57, y=26), bottom_right=Point(x=58, y=26))